## For GC_Selvarajan: 
- attention has duplicates
- variant map, vcf and region bed

### Process:
- split the names by # => multiple sequences with different headers 
- 

In [8]:
from importlib import reload
import pandas as pd
import sys
import os
sys.path.append('../helpful_functions')
import helpful_functions as hf
reload(hf)

<module 'helpful_functions' from '/home/kisa/coding/80K_MPRA/80K-Analysis/07_quality_control/notebooks/control_metadata/../helpful_functions/helpful_functions.py'>

In [9]:
# column names
col_name = 'name'
col_header = 'name'
col_sequence = 'sequence'
col_category = 'category'
col_class = 'class'
col_source = 'source'
col_ref = 'ref'
col_chr = 'chr'
col_start = 'start'
col_end = 'end'
col_strand = 'strand'
col_variant_class = 'variant_class'
col_variant_pos = 'variant_pos'
col_SPDI = 'SPDI'
col_allele = 'allele'
col_info = 'info'
my_col_ref_base = 'tmp_ref_base'
my_col_alt_base = 'tmp_alt_base'


interesting_columns = [col_name, col_sequence, col_category, col_class, col_source, col_ref,
                       col_chr, col_start, col_end, col_strand, col_variant_class, col_variant_pos, col_SPDI, col_allele, col_info]

In [10]:
import yaml

# config
config_path = "/home/kisa/coding/80K_MPRA/80K-Analysis/07_quality_control/notebooks/control_metadata/config_file.yaml"
with open(config_path) as conf:
    config = yaml.load(conf, Loader=yaml.FullLoader)
    conf.close()

input_fasta = config['design_file']
pre_metadata_df = hf.fasta_to_dataframe(input_fasta, columns=[col_name, col_sequence])
pre_metadata_df['tmp_label'] = pre_metadata_df[col_name].apply(lambda x: hf.get_label(x))

# variant map
variant_map_path = '/home/kisa/coding/80K_MPRA/design_data/design_info/renamed_variant_region_map_unique.tsv.gz'
variant_map_path = config['variant_region_map']
variant_map = pd.read_csv(variant_map_path, sep="\t")
# variant_map.columns = ['ID', 'Region', 'REF', 'ALT']
# variant_map.to_csv('/home/kisa/coding/80K_MPRA/design_data/design_info/variant_region_map_new_colnames.tsv.gz', sep="\t", compression='gzip', index=False)
variant_map['tmp_label'] = variant_map['ID'].apply(hf.get_label)

# region bed
region_bed = '/home/kisa/coding/80K_MPRA/design_data/design_info/renamed_regions.bed.gz'
region_bed = config['region_bed']
region_bed = pd.read_csv(region_bed, sep="\t", header=None)
region_bed.columns = [f'region_{col_name}' for col_name in ['chr', 'start', 'end', 'name', 'score', 'strand']]

# vcf
vcf_path = '/home/kisa/coding/80K_MPRA/design_data/design_info/variants.vcf.gz'
vcf_path = config['variant_vcf']
vcf_df = pd.read_csv(vcf_path, sep='\t', comment="#", header=None)
vcf_df.columns = ['CHROM', 'var_pos', 'ID', 'vcf_REF', 'vcf_ALT', 'QUAL', 'FILTER', 'INFO']


In [ ]:
# rename the sequences:
# - GC_Mendelian_variants:REF_chr8:11703890AG>A|GATA4#GC_Mendelian_variants:ALT_chr8:11703890AG>A|GATA4_chr8:11703890AG>A|GATA4 > TMP_HEADER_TO_BE_CHANGED_TO_REF
# - GC_Mendelian_variants:REF_chr8:11703890AG>A|GATA4 > GC_Mendelian_variants:ALT_chr8:11703890AG>A|GATA4_chr8:11703890AG>A|GATA4
# - TMP_HEADER_TO_BE_CHANGED_TO_REF > GC_Mendelian_variants:REF_chr8:11703890AG>A|GATA4
# - GC_Mendelian_variants:REF_chr8:11703860G>T|GATA4#GC_Mendelian_variants:ALT_chr8:11703860G>T|GATA4_chr8:11703890AG>A|GATA4 > TMP_HEADER_TO_BE_CHANGED_TO_REF
# - GC_Mendelian_variants:REF_chr8:11703860G>T|GATA4 > GC_Mendelian_variants:ALT_chr8:11703860G>T|GATA4_chr8:11703890AG>A|GATA4
# - TMP_HEADER_TO_BE_CHANGED_TO_REF > GC_Mendelian_variants:REF_chr8:11703860G>T|GATA4
# done in the preprocessing: /home/kisa/coding/80K_MPRA/design_data/design_info/design_removed_spaces_deduplicated_sequences_renamed.fa resulting


In [ ]:
# 'GC_Mendelian_variants:ALT_chr8:11703890AG>A|GATA4_chr8:11703890AG>A|GATA4' in rename_map
# "GC_Mendelian_variants:REF_chr8:11703890AG>A|GATA4" in rename_map

True

In [ ]:
groups_with_duplicates = ['C_negative_heart_MK',
 'C_negative_neuron_MK',
 'C_positive_heart_CAD',
 'C_positive_heart_MK',
 'C_positive_neuron_CD',
 'C_positive_neuron_MK',
 'GC_GABA_Chengyu',
 'GC_Glut_Chengyu',
 'GC_Mendelian_variants',
 'GC_Mohlke',
 'GC_Selvarajan',
 'MK']